# THEMIS v3 — LoRA Fine-tuning on Kaggle T4

**Base Model:** Mistral 7B Instruct v0.3 (4-bit NF4 quantized)
**Method:** LoRA rank=16, alpha=32, all attention modules (q, k, v, o)
**Dataset:** ~20,909 Indian law Q&A pairs (BNS, BNSS, BSA, CPA, RTI Act)
**Key Change from v2:** 2 epochs instead of 3 — fixes overfitting (loss 0.06→0.08 memorization)

---

## What went wrong in v2 and what this fixes

| Problem | v2 behavior | v3 fix |
|---------|-------------|--------|
| Overfitting | Loss dropped to 0.06-0.08, model memorized training patterns | 2 epochs instead of 3 |
| Regurgitation | Model recited statute definitions verbatim instead of answering the question | Early stopping via lower epoch count |
| Repetition loops | Disclaimer text repeated 2x, cut off at token limit | Reduced epochs prevent memorizing boilerplate |
| No checkpoints saved | Intermediate checkpoints lost when Kaggle session ended | Download checkpoints after training |

---

## Prerequisites — Do These First

### Step 1: Get a HuggingFace Token

1. Go to **https://huggingface.co/settings/tokens**
2. Click **"New token"**
3. Name: `kaggle-training` (or anything)
4. Role: **"Write"** (needed to push adapters)
5. Click **"Generate token"**
6. **Copy the token** — it looks like `hf_AbCdEfGhIjKlMnOpQrStUvWxYz123456789`

### Step 2: Add Token as Kaggle Secret

1. On this notebook page, look at the **right sidebar**
2. Click **"Settings"** tab
3. Scroll to **"Secrets"** section
4. Click **"Add secret"**
5. Fill in:
   - **Name:** `HF_TOKEN` (must be exactly this, all caps)
   - **Value:** paste your HuggingFace token from Step 1
6. Click **"Add"**

### Step 3: Enable GPU

1. Still in **Settings** sidebar
2. Find **"Accelerator"** section
3. Select **"GPU T4 x2"** (or GPU T4 x1 if x2 not available)
4. If prompted, click **"Turn on internet"** (needed to download model from HuggingFace)

### Step 4: Upload Training Data

1. In the **right sidebar**, click **"Data"** tab
2. Click **"+ Add Data"**
3. Search for your dataset or click **"Upload"**
4. Upload `dataset.json` from your local machine
5. Note the path it appears at (usually `/kaggle/input/your-dataset-name/dataset.json`)

---

**After completing all 4 steps above, run each cell below in order.**

## Cell 1 — Install Dependencies

This installs the training libraries we need:

- **unsloth**: 2x faster LoRA training with 60% less VRAM usage. Critical for fitting 7B model on T4.
- **trl**: HuggingFace's training library. Provides `SFTTrainer` which handles the training loop.
- **transformers**: HuggingFace model/tokenizer loading.
- **peft**: Parameter-Efficient Fine-Tuning — the library that implements LoRA.
- **datasets**: HuggingFace dataset handling.
- **accelerate**: Optimizes training across devices.
- **bitsandbytes**: 4-bit quantization — reduces model from ~14GB to ~4GB.

**First run takes ~2-3 minutes.** Subsequent runs skip if already installed (the `%%capture` hides the pip output).

In [ ]:
%%capture
!pip install unsloth
!pip install --upgrade --no-deps trl transformers datasets peft accelerate bitsandbytes

## Cell 2 — Verify GPU is Available

Training a 7B parameter model requires a GPU. This cell checks:
- Is CUDA available?
- What GPU do we have? (Should be NVIDIA T4)
- How much VRAM? (T4 has ~15GB)

**If this fails:** Go to Settings → Accelerator → select GPU T4 x2.

**Do not proceed without a GPU** — training on CPU would take weeks.

In [ ]:
import torch

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_mem / 1e9
    print(f"GPU Found: {gpu_name}")
    print(f"VRAM: {vram_gb:.1f} GB")
    
    if vram_gb < 14:
        print("\nWARNING: Low VRAM. Training may fail or be slow.")
        print("Try: Settings → Accelerator → GPU T4 x2")
    else:
        print("\nVRAM sufficient. Ready to train.")
else:
    raise RuntimeError(
        "No GPU found!\n"
        "Go to Settings (right sidebar) → Accelerator → GPU T4 x2\n"
        "Then restart the notebook kernel and run this cell again."
    )

## Cell 3 — Load Base Model in 4-bit Quantization

We load **Mistral 7B Instruct v0.3** using BitsAndBytes 4-bit NF4 quantization.

### What is 4-bit quantization?

The full model in fp16 would need ~14GB of VRAM — more than a T4 has. 4-bit NF4 quantization compresses the model weights to ~4GB while preserving most of the model's capabilities. This leaves ~11GB of VRAM for training.

### What is Unsloth's FastLanguageModel?

Unsloth wraps HuggingFace's model loading with optimizations:
- Automatic 4-bit quantization
- Gradient checkpointing (trades compute for memory)
- Flash attention 2 (faster, less memory)
- 2x faster training vs standard HuggingFace

**First run downloads ~4GB from HuggingFace.** Takes 2-5 minutes depending on internet speed.

In [ ]:
from unsloth import FastLanguageModel

# ============================================================
# CONFIGURATION — Change these if needed
# ============================================================
BASE_MODEL = "unsloth/mistral-7b-instruct-v0.3-bnb-4bit"  # Pre-quantized 4-bit model
MAX_SEQ_LENGTH = 1024  # Max tokens per sample. BNS sections can be long.
# ============================================================

print(f"Loading model: {BASE_MODEL}")
print(f"Max sequence length: {MAX_SEQ_LENGTH} tokens")
print("This may take 2-5 minutes on first run (downloading ~4GB)...\n")

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,           # Auto-detect: fp16 on T4 GPU
    load_in_4bit=True,    # NF4 quantization — critical for T4 VRAM
)

print(f"\nModel loaded successfully.")
print(f"Tokenizer vocab size: {tokenizer.vocab_size:,}")

## Cell 4 — Add LoRA Adapters to the Model

### What is LoRA?

LoRA (Low-Rank Adaptation) is the core technique. Instead of updating all 7 billion parameters (which would require ~56GB VRAM), we:

1. Freeze the original model weights
2. Add small trainable matrices (adapters) to each attention layer
3. Only train these adapters (~0.1% of parameters)

### v3 LoRA Configuration

| Parameter | v1 | v2 | v3 | Why |
|-----------|-----|-----|-----|-----|
| Rank (r) | 8 | 16 | 16 | Adapter capacity — 16 captures legal patterns well |
| Alpha | 16 | 32 | 32 | Scaling factor — alpha/rank = 2x scaling |
| Target modules | q, v | q, k, v, o | q, k, v, o | All 4 attention projections |
| Dropout | 0 | 0.05 | 0.05 | Regularization — prevents overfitting |

### Trainable Parameters

With rank=16 and all 4 attention modules:
- Trainable: ~8.4M parameters
- Total: ~7.3B parameters
- Percentage: ~0.12%

This is why LoRA is so efficient — we train 0.12% of parameters and get domain knowledge.

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,                                  # LoRA rank — controls adapter capacity
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],  # All attention layers
    lora_alpha=32,                         # Scaling factor (2x rank)
    lora_dropout=0.05,                     # Regularization — helps prevent overfitting
    bias="none",                           # No bias terms in LoRA adapters
    use_gradient_checkpointing=True,       # Saves VRAM by recomputing activations
    random_state=42,                       # Reproducibility
)

# Count trainable parameters
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())

print(f"LoRA adapters added.")
print(f"Trainable parameters: {trainable_params:,}")
print(f"Total parameters:     {total_params:,}")
print(f"Trainable %:          {100 * trainable_params / total_params:.2f}%")

## Cell 5 — Load the Training Dataset

### Dataset Format (Alpaca JSON)

Each training sample is a JSON object with three fields:

```json
{
  "instruction": "What is the punishment for theft under BNS?",
  "input": "",
  "output": "Under Section 303 of the Bharatiya Nyaya Sanhita, 2023..."
}
```

- **instruction**: The legal question being asked
- **input**: Additional context (usually empty for simple Q&A)
output**: The correct answer with section citations and disclaimer

### How to Upload Your Dataset

**Option A: Kaggle Dataset (Recommended)**
1. Go to kaggle.com → Datasets → New Dataset
2. Upload `dataset.json` (from `themis/data/dataset.json`)
3. Copy the dataset URL
4. Update `DATASET_PATH` below

**Option B: Direct Upload**
1. Click the "+ Data" button in the right sidebar
2. Click "Upload" → select `dataset.json`
3. Note the path it appears at

**After upload, update `DATASET_PATH` below to match your file location.**

In [ ]:
import json
from datasets import Dataset

# ============================================================
# UPDATE THIS PATH to match where your dataset.json is
# ============================================================
# Option A: If you created a Kaggle Dataset
DATASET_PATH = "/kaggle/input/themis-dataset/dataset.json"

# Option B: If you uploaded directly to the notebook
# DATASET_PATH = "/kaggle/input/dataset.json"
# ============================================================

print(f"Loading dataset from: {DATASET_PATH}")

try:
    with open(DATASET_PATH, "r", encoding="utf-8") as f:
        raw_data = json.load(f)
except FileNotFoundError:
    raise FileNotFoundError(
        f"Dataset not found at {DATASET_PATH}\n"
        f"\n"
        f"Fix: Upload dataset.json using one of these methods:\n"
        f"  1. Click '+ Data' in right sidebar → Upload\n"
        f"  2. Create a Kaggle Dataset and update DATASET_PATH above"
    )

print(f"\nLoaded {len(raw_data):,} training samples")
print(f"\nSample instruction: {raw_data[0]['instruction'][:80]}...")
print(f"Sample output length: {len(raw_data[0]['output'])} chars")

# Show distribution of output lengths
lengths = [len(s['output']) for s in raw_data]
print(f"\nOutput length stats:")
print(f"  Min: {min(lengths)} chars")
print(f"  Max: {max(lengths)} chars")
print(f"  Avg: {sum(lengths)//len(lengths)} chars")

# Create HuggingFace Dataset
dataset = Dataset.from_list(raw_data)
print(f"\nDataset created: {len(dataset):,} samples")

## Cell 6 — Preview the Training Format

This shows exactly what the model will see during training. Each sample is formatted as:

```
### Instruction:
{the legal question}

### Response:
{the answer with section citations}
```

The model learns to predict the Response given the Instruction. This is standard SFT (Supervised Fine-Tuning).

In [ ]:
def format_instruction(sample):
    """Convert a JSON sample into the Alpaca prompt format."""
    text = f"### Instruction:\n{sample['instruction']}\n\n"
    if sample.get("input"):
        text += f"### Input:\n{sample['input']}\n\n"
    text += f"### Response:\n{sample['output']}"
    return text

# Show a formatted example
print("=" * 70)
print("EXAMPLE: What the model sees during training")
print("=" * 70)
example = format_instruction(raw_data[0])
print(example)
print("=" * 70)

# Token count
token_count = len(tokenizer.encode(example))
print(f"\nToken count: {token_count}")
print(f"Max allowed: {MAX_SEQ_LENGTH}")
if token_count > MAX_SEQ_LENGTH:
    print(f"WARNING: Sample exceeds max_seq_length! It will be truncated.")
else:
    print(f"OK: Sample fits within max_seq_length.")

## Cell 7 — Configure Training Hyperparameters

### v3 Training Config (Fixed from v2)

| Parameter | v2 | v3 | Why Changed |
|-----------|-----|-----|-------------|
| **Epochs** | **3** | **2** | **v2 overfit at 3 epochs. 2 epochs prevents memorization.** |
| Batch size | 1 | 1 | T4 VRAM limit — can't go higher |
| Grad accum | 8 | 8 | Effective batch = 8 samples per step |
| Learning rate | 2e-4 | 2e-4 | Proven stable for LoRA |
| Max seq len | 1024 | 1024 | Covers most statute sections |
| Warmup | 3% | 3% | Stable warmup |
| Optimizer | adamw_8bit | adamw_8bit | VRAM efficient |
| Save steps | 500 | 500 | Checkpoint every 500 steps |
| Save limit | 2 | 3 | Keep more checkpoints to find best one |

### Estimated Training Time

- Dataset: ~20,909 samples
- Effective batch size: 8 (1 × 8 grad accum)
- Steps per epoch: 20,909 ÷ 8 ≈ 2,614 steps
- Total steps: 2,614 × 2 epochs ≈ **5,228 steps**
- Time per step: ~1.5 seconds on T4
- **Total time: ~2.2 hours**

### What to Watch During Training

- **Loss should decrease** from ~2.0 to ~0.3-0.5
- **Good sign**: Loss stabilizes around 0.15-0.25 by end of epoch 2
- **Bad sign**: Loss drops below 0.10 — model is memorizing, not learning
- **Bad sign**: Loss spikes above 5.0 — learning rate too high (unlikely with these settings)

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

# ============================================================
# TRAINING CONFIGURATION
# ============================================================
NUM_EPOCHS = 2                    # KEY CHANGE: 2 instead of 3 to prevent overfitting
BATCH_SIZE = 1                    # Per-device batch size (T4 VRAM limit)
GRAD_ACCUM = 8                    # Gradient accumulation steps
LEARNING_RATE = 2e-4              # Learning rate
WARMUP_RATIO = 0.03               # 3% warmup steps
MAX_SEQ_LEN = 1024                # Max tokens per sample
SAVE_STEPS = 500                  # Save checkpoint every 500 steps
SAVE_TOTAL_LIMIT = 3              # Keep last 3 checkpoints (more than v2's 2)
OUTPUT_DIR = "themis-v3-outputs"   # Where checkpoints are saved
# ============================================================

effective_batch = BATCH_SIZE * GRAD_ACCUM
steps_per_epoch = len(dataset) // effective_batch
total_steps = steps_per_epoch * NUM_EPOCHS

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    formatting_func=format_instruction,
    max_seq_length=MAX_SEQ_LEN,
    args=TrainingArguments(
        per_device_train_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM,
        warmup_ratio=WARMUP_RATIO,
        num_train_epochs=NUM_EPOCHS,
        learning_rate=LEARNING_RATE,
        fp16=True,                         # T4 uses fp16, not bf16
        logging_steps=10,                  # Print loss every 10 steps
        output_dir=OUTPUT_DIR,
        optim="adamw_8bit",                # Memory-efficient optimizer
        seed=42,                           # Reproducibility
        save_strategy="steps",
        save_steps=SAVE_STEPS,             # Checkpoint every 500 steps
        save_total_limit=SAVE_TOTAL_LIMIT, # Keep last 3 checkpoints
    ),
)

print("Trainer configured.")
print(f"Epochs: {NUM_EPOCHS}")
print(f"Effective batch size: {effective_batch}")
print(f"Steps per epoch: ~{steps_per_epoch:,}")
print(f"Total steps: ~{total_steps:,}")
print(f"Checkpoints saved every: {SAVE_STEPS} steps")
print(f"Keep last: {SAVE_TOTAL_LIMIT} checkpoints")
print(f"\nEstimated training time: ~{total_steps * 1.5 / 3600:.1f} hours")

## Cell 8 — Run Training

This is the main training loop. The model will:

1. **Epoch 1** (~2,614 steps): Learning legal domain patterns
2. **Epoch 2** (~2,614 steps): Reinforcing patterns, refining knowledge

### What You'll See

Every 10 steps, the trainer prints:
- `step`: Current step number
- `training_loss`: How well the model is learning (lower = better)
- `runtime`: Time elapsed
- `tokens_per_sec`: Training speed

### Important

- **Do NOT interrupt** once training starts — you'll lose progress
- **Runtime:** ~2-2.5 hours on T4
- **Checkpoints** are saved every 500 steps to `themis-v3-outputs/`
- **If Kaggle disconnects:** You can reload from the last checkpoint (Cell 8b)

In [ ]:
import time

start_time = time.time()

print("=" * 70)
print("THEMIS v3 Training Started")
print("=" * 70)
print(f"Model: {BASE_MODEL}")
print(f"Dataset: {len(dataset):,} samples")
print(f"Epochs: {NUM_EPOCHS}")
print(f"Effective batch size: {effective_batch}")
print(f"Total steps: ~{total_steps:,}")
print(f"Estimated time: ~{total_steps * 1.5 / 3600:.1f} hours")
print("=" * 70)
print("\nTraining in progress...\n")

trainer_stats = trainer.train()

elapsed = time.time() - start_time
hours = int(elapsed // 3600)
minutes = int((elapsed % 3600) // 60)

print("\n" + "=" * 70)
print("TRAINING COMPLETE")
print("=" * 70)
print(f"Time: {hours}h {minutes}m")
print(f"Final training loss: {trainer_stats.training_loss:.4f}")
print(f"Total steps: {trainer_stats.global_step}")
print("=" * 70)

## Cell 8b — Resume from Checkpoint (Only if Needed)

**Skip this cell if training completed successfully above.**

Use this only if:
- Kaggle disconnected during training
- You want to continue training from a saved checkpoint

To find your checkpoints, run `!ls themis-v3-outputs/` and look for `checkpoint-XXXX` folders.

In [ ]:
# ============================================================
# ONLY RUN IF TRAINING WAS INTERRUPTED
# Uncomment the lines below and set CHECKPOINT_PATH
# ============================================================
# CHECKPOINT_PATH = "themis-v3-outputs/checkpoint-2500"  # Update this
# 
# import time
# start_time = time.time()
# print(f"Resuming from checkpoint: {CHECKPOINT_PATH}")
# 
# trainer_stats = trainer.train(resume_from_checkpoint=CHECKPOINT_PATH)
# 
# elapsed = time.time() - start_time
# hours = int(elapsed // 3600)
# minutes = int((elapsed % 3600) // 60)
# print(f"\nResumed training complete in {hours}h {minutes}m")
# print(f"Final loss: {trainer_stats.training_loss:.4f}")

## Cell 9 — Analyze Training Results

After training, we check the loss curve to verify the model didn't overfit.

### What to Look For

| Loss Range | Interpretation |
|------------|----------------|
| 0.3 - 0.5 | Good — model learned without overfitting |
| 0.15 - 0.3 | Excellent — strong learning, minimal overfitting |
| 0.06 - 0.1 | **Overfitting** — model memorized training data |
| < 0.05 | **Severe overfitting** — model is pure memorization |
| > 0.5 | Underfitting — model didn't learn enough |

In [ ]:
# Extract training metrics
log_history = trainer_state.log_history if hasattr(trainer, 'state') else []

if trainer_stats:
    final_loss = trainer_stats.training_loss
    total_steps = trainer_stats.global_step
    
    print("Training Summary:")
    print(f"  Final loss: {final_loss:.4f}")
    print(f"  Total steps: {total_steps}")
    
    if final_loss < 0.1:
        print("\nWARNING: Loss below 0.1 — possible overfitting!")
        print("Consider using an earlier checkpoint (check themis-v3-outputs/)")
    elif final_loss < 0.2:
        print("\nGOOD: Loss in healthy range (0.1-0.2). Model learned well.")
    elif final_loss < 0.5:
        print("\nGOOD: Loss in acceptable range (0.2-0.5). Model has room to improve.")
    else:
        print("\nWARNING: Loss above 0.5 — model may need more training or different hyperparameters.")
else:
    print("No training stats available.")

## Cell 10 — Save the Trained Adapter

Save the LoRA adapter (the small trained weight deltas) to the Kaggle working directory.

### What Gets Saved

- `adapter_model.bin`: The trained LoRA weights (~30MB)
- `adapter_config.json`: LoRA configuration
- `tokenizer.json` + `tokenizer.model`: Tokenizer files

### Why This Matters

The adapter is small (~30MB) compared to the full model (~4GB). To run inference, you load the base model + this adapter.

In [ ]:
from pathlib import Path

ADAPTER_DIR = "./themis-lora-v3"
Path(ADAPTER_DIR).mkdir(parents=True, exist_ok=True)

print(f"Saving adapter to: {ADAPTER_DIR}")

model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)

# List saved files
adapter_files = list(Path(ADAPTER_DIR).glob("*"))
total_size = sum(f.stat().st_size for f in adapter_files if f.is_file())

print(f"\nAdapter saved successfully.")
print(f"Total size: {total_size / 1e6:.1f} MB")
print(f"\nFiles saved:")
for f in sorted(adapter_files):
    if f.is_file():
        print(f"  {f.name} ({f.stat().st_size / 1e6:.2f} MB)")

## Cell 11 — Push Adapter to HuggingFace Hub

Upload the adapter to HuggingFace Hub so it can be loaded from anywhere (your local machine, other notebooks, etc.).

### This Uses Your HF_TOKEN Secret

You configured this in the prerequisites. If you see an error, check that:
1. The secret name is exactly `HF_TOKEN` (all caps)
2. The token has **Write** permission
3. You haven't revoked the token

### Repository

The adapter will be pushed to: `Daniel2503/themis-mistral-7b-lora-v3`

If this repo doesn't exist, it will be created automatically.

In [ ]:
import os
from huggingface_hub import HfApi, login

# ============================================================
# HuggingFace Authentication
# ============================================================
# This reads from the HF_TOKEN Kaggle Secret you configured
HF_TOKEN = os.environ.get("HF_TOKEN")

if not HF_TOKEN:
    raise RuntimeError(
        "HF_TOKEN not found!\n"
        "\n"
        "Fix: Add it as a Kaggle Secret:\n"
        "  1. Right sidebar → Settings tab\n"
        "  2. Scroll to 'Secrets' section\n"
        "  3. Click 'Add secret'\n"
        "  4. Name: HF_TOKEN (all caps)\n"
        "  5. Value: your HuggingFace token (hf_...)\n"
        "  6. Click 'Add'\n"
        "  7. RESTART KERNEL and run cells from the beginning"
    )

login(token=HF_TOKEN)
print("Logged into HuggingFace Hub.")

# ============================================================
# Upload Adapter
# ============================================================
REPO_NAME = "Daniel2503/themis-mistral-7b-lora-v3"

print(f"\nUploading adapter to: {REPO_NAME}")
print("This may take 1-2 minutes...\n")

api = HfApi()
api.upload_folder(
    folder_path=ADAPTER_DIR,
    repo_id=REPO_NAME,
    repo_type="model",
)

print(f"Upload complete!")
print(f"Adapter available at: https://huggingface.co/{REPO_NAME}")

## Cell 12 — Test the Trained Model

Load the base model + your trained adapter and test with a sample legal question.

### Test Questions

We test with both:
1. **Direct question** (similar to training format): "What is the punishment for theft under BNS?"
2. **Conversational question** (rephrased): "If someone steals property worth ₹5000, what charges apply?"

The second question tests whether the model can handle natural language, not just template phrases.

In [ ]:
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

print("Loading base model + trained adapter for testing...")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
)

test_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)

test_tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
test_model = PeftModel.from_pretrained(test_model, ADAPTER_DIR)
test_model.eval()

print("Model loaded. Running test queries...\n")

def ask_themis(question, max_tokens=256):
    """Run a question through the trained model."""
    prompt = f"### Instruction:\n{question}\n\n### Response:\n"
    inputs = test_tokenizer(prompt, return_tensors="pt").to("cuda")
    
    with torch.no_grad():
        outputs = test_model.generate(
            **inputs,
            max_new_tokens=max_tokens,
            temperature=0.3,
            top_p=0.9,
            repetition_penalty=1.1,
            do_sample=True,
        )
    
    response = test_tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    )
    return response

# Test 1: Direct question (training format)
print("=" * 70)
print("TEST 1: Direct question (similar to training data)")
print("=" * 70)
q1 = "What is the punishment for theft under the Bharatiya Nyaya Sanhita?"
print(f"Q: {q1}")
print(f"\nA: {ask_themis(q1)}")

# Test 2: Conversational question (rephrased)
print("\n" + "=" * 70)
print("TEST 2: Conversational question (rephrased — tests generalization)")
print("=" * 70)
q2 = "If someone steals property worth 5000 rupees, what charges can be filed against them?"
print(f"Q: {q2}")
print(f"\nA: {ask_themis(q2)}")

# Test 3: Out-of-scope question (should refuse)
print("\n" + "=" * 70)
print("TEST 3: Out-of-scope question (should refuse)")
print("=" * 70)
q3 = "What is the capital of France?"
print(f"Q: {q3}")
print(f"\nA: {ask_themis(q3)}")

## Cell 13 — Download Adapter to Local Machine

Download the trained adapter as a zip file for local use.

### How to Download

1. Run the cell below to create a zip file
2. In Kaggle's **Output** panel (right sidebar), find `themis-lora-v3.zip`
3. **Right-click** → **Download**
4. Extract to `D:\Vs Code\themis\model\themis-lora\` on your local machine

### Also Download Checkpoints!

If you want to compare different training checkpoints:
1. Run `!ls themis-v3-outputs/` to see available checkpoints
2. Zip and download the ones you want to test

In [ ]:
import shutil

# Zip the final adapter
shutil.make_archive("themis-lora-v3", "zip", ".", ADAPTER_DIR)
print("Adapter zipped as: themis-lora-v3.zip")

# List checkpoints available
print("\nCheckpoints saved during training:")
checkpoint_dirs = sorted(Path(OUTPUT_DIR).glob("checkpoint-*"))
if checkpoint_dirs:
    for cp in checkpoint_dirs:
        cp_size = sum(f.stat().st_size for f in cp.rglob("*") if f.is_file())
        print(f"  {cp.name} ({cp_size / 1e6:.1f} MB)")
    
    print("\nTo download a specific checkpoint:")
    print("  1. Run: !zip -r checkpoint-XXXX.zip themis-v3-outputs/checkpoint-XXXX")
    print("  2. Download from Kaggle Output panel")
else:
    print("  No checkpoints found (training may not have reached first save point)")

print("\nTo download the final adapter:")
print("  Right-click 'themis-lora-v3.zip' in Kaggle Output panel → Download")
print(f"\nAlso available at: https://huggingface.co/{REPO_NAME}")

## Cell 14 — Quick Comparison: v2 vs v3

If you have both v2 and v3 adapters, run this to compare them side-by-side.

**Skip this cell if you don't have the v2 adapter available.**

In [ ]:
# ============================================================
# COMPARISON: v2 vs v3 (uncomment if you have both adapters)
# ============================================================
# 
# V2_ADAPTER = "./themis-lora-v2"  # Path to v2 adapter
# V3_ADAPTER = "./themis-lora-v3"  # Path to v3 adapter
# 
# def load_model(adapter_path):
#     """Load base model with a specific adapter."""
#     model = AutoModelForCausalLM.from_pretrained(
#         BASE_MODEL,
#         quantization_config=bnb_config,
#         device_map="auto",
#     )
#     tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
#     model = PeftModel.from_pretrained(model, adapter_path)
#     model.eval()
#     return model, tokenizer
# 
# test_questions = [
#     "What is the punishment for theft under BNS?",
#     "If someone steals property worth 5000 rupees, what charges apply?",
#     "What happens if someone is caught with fake documents?",
# ]
# 
# print("COMPARISON: v2 vs v3")
# print("=" * 70)
# 
# for q in test_questions:
#     print(f"\nQ: {q}")
#     print("-" * 40)
#     
#     # Test v2
#     v2_model, v2_tok = load_model(V2_ADAPTER)
#     prompt = f"### Instruction:\n{q}\n\n### Response:\n"
#     inputs = v2_tok(prompt, return_tensors="pt").to("cuda")
#     with torch.no_grad():
#         out = v2_model.generate(**inputs, max_new_tokens=256, temperature=0.3)
#     v2_response = v2_tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
#     print(f"v2: {v2_response[:200]}...")
#     del v2_model  # Free VRAM
#     
#     # Test v3
#     v3_model, v3_tok = load_model(V3_ADAPTER)
#     inputs = v3_tok(prompt, return_tensors="pt").to("cuda")
#     with torch.no_grad():
#         out = v3_model.generate(**inputs, max_new_tokens=256, temperature=0.3)
#     v3_response = v3_tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
#     print(f"v3: {v3_response[:200]}...")
#     del v3_model  # Free VRAM
#     
#     print()

## Summary

### What This Notebook Did

1. **Fixed v2 overfitting** by reducing epochs from 3 to 2
2. **Saved checkpoints** every 500 steps (keep last 3)
3. **Tested with conversational questions** to verify generalization
4. **Pushed adapter** to HuggingFace Hub as `themis-mistral-7b-lora-v3`

### Expected Results

| Metric | v2 | v3 (target) |
|--------|-----|-------------|
| Training loss | 0.06-0.08 | 0.15-0.25 |
| Hallucination rate | Low (fixed) | Low |
| Regurgitation | High | Low |
| Conversational Q&A | Poor | Good |

### Next Steps

1. Download adapter from HuggingFace or Kaggle Output
2. Run `themis ask "your question"` locally to test
3. Run full eval: `themis eval`
4. Compare v2 vs v3 on the expanded eval set